In [ ]:
# Consome do Kafka o tópico gerado pelo CDC da tabela pedidos e salva no MinIO.
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

# 1. Inicialização do Spark carregando o pacote do Kafka e do S3/MinIO
spark = SparkSession.builder \
    .appName("SparkStreamingCDC") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,org.apache.hadoop:hadoop-aws:3.3.4") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "root-minio") \
    .config("spark.hadoop.fs.s3a.secret.key", "root12345678") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

In [ ]:
# O Debezium cria o tópico no padrão: prefixo.schema.tabela -> cdc.public.pedidos
df_kafka_bruto = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "cdc.public.pedidos") \
    .option("startingOffsets", "earliest") \
    .load()

# No Kafka, a mensagem vem binária na coluna 'value'
df_kafka_string = df_kafka_bruto.selectExpr("CAST(value AS STRING) as json_str")

In [ ]:
# O Debezium encapsula o dado real dentro de um bloco chamado 'after' (pós-mudança)
schema_debezium = StructType([
    StructField("payload", StructType([
        StructField("after", StructType([
            StructField("id_pedido", StringType(), True),
            StructField("id_cliente", StringType(), True),
            StructField("data_pedido", LongType(), True), # Timestamptz vem convertido em milissegundos
            StructField("status", StringType(), True),
            StructField("total_pedido", StringType(), True) # Numerics costumam vir em string ou bytes no CDC
        ]), True),
        StructField("op", StringType(), True) # 'c' para insert/create, 'u' para update
    ]), True)
])

In [ ]:
# Parsing/Processamento do JSON "em voo"
df_parsed = df_kafka_string \
    .withColumn("dados_origem", from_json(col("json_str"), schema_debezium)) \
    .select("dados_origem.payload.after.*", "dados_origem.payload.op") \
    .filter(col("id_pedido").isNotNull()) # Filtra eventuais deletes (onde after é nulo)

In [ ]:
# Escrita contínua (Streaming Sink) salvando no MinIO
# Usa checkpointing para garantir resiliência (Exactly-Once Semantics)
query_minio = df_parsed.writeStream \
    .format("parquet") \
    .option("checkpointLocation", "s3a://datalake/checkpoints/pedidos_streaming/") \
    .option("path", "s3a://datalake/live/pedidos/") \
    .outputMode("append") \
    .start()

print("Pipeline de Streaming executando e aguardando dados...")
query_minio.awaitTermination()

In [ ]:
# Verifica se o streaming ainda está rodando e mostra o último status
print(f"O streaming está ativo? {query_minio.isActive}")
if query_minio.isActive:
    print("Último progresso:", query_minio.lastProgress)

In [ ]:
# Encerra o streaming e a sessão do Spark de forma limpa
print("Parando o streaming de dados...")
query_minio.stop()

print("Fechando a sessão do Spark...")
spark.stop()

print("Ambiente finalizado com sucesso!")